# SplaTAM trên dữ liệu đồ án (Kaggle, GPU T4, bật Internet)

Notebook riêng, **không** cài môi trường MonoGS ở đây: rasterizer của SplaTAM trùng tên module `diff_gaussian_rasterization` với của MonoGS.
Notebook chỉ chạy SLAM và xuất kết quả (`<work>/export/`). Bước chấm (`baselines/common/score.py`) chạy sau, trong môi trường MonoGS
(notebook `run-monogs` hoặc WSL), xem `baselines/methods/splatam/README.md`.

In [ ]:
# Repo đồ án: chỉ cần configs/rgbd/replica/base_config.yaml (nội tại, depth_scale) và baselines/.
!git clone --depth 1 https://github.com/Trung4n/LunoGS
# baselines/ nằm trong .gitignore của repo: chép từ Kaggle dataset (hoặc repo riêng) vào LunoGS/baselines.
BASELINES_SRC = "<ĐIỀN: /kaggle/input/<dataset>/baselines>"
!rm -rf LunoGS/baselines && cp -r {BASELINES_SRC} LunoGS/baselines && ls LunoGS/baselines/methods/splatam

In [ ]:
%%time
# SplaTAM, ghim commit (ghi vào metrics.json qua --method_commit).
SPLATAM_COMMIT = "da6bbcd24c248dc884ac7f49d62e91b841b26ccc"
!git clone -q https://github.com/spla-tam/SplaTAM.git && git -C SplaTAM checkout -q {SPLATAM_COMMIT} && git -C SplaTAM log -1 --format='%H %ci'

# Phụ thuộc trên đường chạy SLAM (scripts/splatam.py và utils/, datasets/ nó import). Torch của Kaggle giữ nguyên
# (README SplaTAM: đã thử với torch 2.3 + CUDA 12.1). open3d, faiss, cyclonedds chỉ dùng cho viz/demo: không cài.
!pip install -q natsort kornia pytorch-msssim torchmetrics
!python -c "import torch, cv2, imageio, yaml, wandb, scipy, natsort, kornia, pytorch_msssim, torchmetrics; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))"

In [ ]:
%%time
# Rasterizer có depth của SplaTAM, đúng commit trong requirements.txt của nó, build cho T4 (sm 7.5).
# Sửa build duy nhất: thêm `#include <cstdint>` nếu thiếu (GCC mới không tự kéo uint32_t); không đổi thuật toán.
DGR_COMMIT = "cb65e4b86bc3bd8ed42174b72a62e8d3a3a71110"
!git clone -q --recursive https://github.com/JonathonLuiten/diff-gaussian-rasterization-w-depth.git dgr && git -C dgr checkout -q {DGR_COMMIT} && git -C dgr submodule update -q --init --recursive
!grep -q "<cstdint>" dgr/cuda_rasterizer/rasterizer_impl.h || sed -i '1i #include <cstdint>' dgr/cuda_rasterizer/rasterizer_impl.h
!TORCH_CUDA_ARCH_LIST="7.5" pip install -q --no-build-isolation ./dgr
!python -c "import diff_gaussian_rasterization, importlib.metadata as m; print(diff_gaussian_rasterization.__file__, m.version('diff_gaussian_rasterization'))"

In [ ]:
OFFICE0_LIT="/kaggle/input/replica-lunogs/office0/lit"
OFFICE0_EVEN="/kaggle/input/replica-lunogs/office0/even"
OFFICE0_AMB="/kaggle/input/datasets/nguyntrungan/office0"

ROOM0_LIT="/kaggle/input/datasets/nguyntrungan/replica-lunogs/room0/lit"
ROOM0_EVEN="/kaggle/input/datasets/nguyntrungan/replica-lunogs/room0/even"

ADAPTER = "LunoGS/baselines/methods/splatam/adapter.py"
CKPT_EVERY = 100  # frame giữa hai checkpoint (0 = tắt); để chạy tiếp khi phiên Kaggle bị ngắt

Mỗi run ghi log vào file (SplaTAM in một thanh tqdm cho mỗi frame, quá dài cho output notebook).
Chạy **tuần tự**, không chạy hai run song song trên hai GPU: FPS phải đo khi máy không bị chia sẻ.

In [ ]:
%%time
WORK = "/kaggle/working/splatam__office0_lit"
!timeout -k 5m 11h python {ADAPTER} run --splatam SplaTAM --dataset {OFFICE0_LIT} --work {WORK} --checkpoint_every {CKPT_EVERY} > {WORK}.log 2>&1; echo exit $?; tail -n 25 {WORK}.log

In [ ]:
%%time
WORK = "/kaggle/working/splatam__office0_even"
!timeout -k 5m 11h python {ADAPTER} run --splatam SplaTAM --dataset {OFFICE0_EVEN} --work {WORK} --checkpoint_every {CKPT_EVERY} > {WORK}.log 2>&1; echo exit $?; tail -n 25 {WORK}.log

**Chạy tiếp sau khi phiên bị ngắt:** thêm output của phiên trước làm input, chép thư mục `WORK` về `/kaggle/working`, rồi chạy cùng lệnh với `--resume`
(frame của checkpoint được xử lý lại một lần; `timing.json` cộng các phiên và ghi rõ là đã resume).

In [ ]:
# WORK = "/kaggle/working/splatam__office0_lit"
# !cp -r /kaggle/input/<output phiên trước>/splatam__office0_lit /kaggle/working/
# !python {ADAPTER} run --splatam SplaTAM --dataset {OFFICE0_LIT} --work {WORK} --checkpoint_every {CKPT_EVERY} --resume >> {WORK}.log 2>&1; tail -n 25 {WORK}.log

In [ ]:
# Dọn output: bỏ checkpoint trung gian, giữ params.npz (bản đồ cuối) và export/ (đầu vào của score.py).
!rm -f /kaggle/working/splatam__*/splatam/*/params[0-9]*.npz /kaggle/working/splatam__*/splatam/*/keyframe_time_indices*.npy
!du -sh /kaggle/working/splatam__*/export /kaggle/working/splatam__*/splatam/*/params.npz; cat /kaggle/working/splatam__*/export/timing.json